# TwoSIDES Preprocessing Pipeline
This notebook processes the TwoSIDES polypharmacy dataset into edge labels for the drug-drug interaction GNN.

**Output:** `twosides_edge_labels.csv` — one row per drug pair × reaction category, with max PRR score as the edge label.

## Step 1: Load and Filter TwoSIDES
Load the raw TwoSIDES CSV in chunks (file is ~42.9M rows), cast PRR to numeric, and filter to statistically significant signals only (PRR > 2).

In [ ]:
import pandas as pd

chunk_size = 500000
chunks = []

print('Loading and filtering TwoSIDES...')
for i, chunk in enumerate(pd.read_csv('TWOSIDES.csv', low_memory=False, chunksize=chunk_size)):
    # Cast PRR to numeric, coerce errors (e.g. string placeholders) to NaN
    chunk['PRR'] = pd.to_numeric(chunk['PRR'], errors='coerce')
    # Keep only statistically significant signals
    chunk = chunk[chunk['PRR'] > 2]
    chunks.append(chunk)
    print(f'  Processed chunk {i+1}, kept {len(chunk):,} rows')

df = pd.concat(chunks, ignore_index=True)
print(f'\nTotal rows after filtering: {len(df):,}')
print(f'Unique drug pairs: {df[["drug_1_rxnorn_id", "drug_2_rxnorm_id"]].drop_duplicates().shape[0]:,}')
print(f'Unique adverse events: {df["condition_meddra_id"].nunique():,}')

## Step 2: Load DrugBank Vocabulary
Load the DrugBank vocabulary file to get the Name → DrugBank ID mapping.

In [ ]:
print('Loading DrugBank vocabulary...')
df_drug = pd.read_csv('drugbank vocabulary.csv')
print(f'DrugBank entries: {len(df_drug):,}')
print(df_drug[['DrugBank ID', 'Name']].head())

## Step 3: Map Drug Names to DrugBank IDs
TwoSIDES uses drug concept names; DrugBank uses its own IDs. We match on lowercase drug name to bridge the two datasets.

This is necessary because TwoSIDES and DrugBank use different identifier systems — without this mapping, the two datasets cannot be joined.

In [ ]:
print('Mapping drug names to DrugBank IDs...')

# Normalise to lowercase for case-insensitive matching
df_drug['name_lower'] = df_drug['Name'].str.lower().str.strip()
df['drug_1_lower'] = df['drug_1_concept_name'].str.lower().str.strip()
df['drug_2_lower'] = df['drug_2_concept_name'].str.lower().str.strip()

# Build name -> DrugBank ID lookup dictionary
name_to_id = dict(zip(df_drug['name_lower'], df_drug['DrugBank ID']))

# Map both drugs in each pair
df['drug_1_drugbank_id'] = df['drug_1_lower'].map(name_to_id)
df['drug_2_drugbank_id'] = df['drug_2_lower'].map(name_to_id)

# Keep only rows where both drugs matched
matched = df['drug_1_drugbank_id'].notna() & df['drug_2_drugbank_id'].notna()
print(f'Rows where BOTH drugs matched: {matched.sum():,} / {len(df):,}')
print(f'Match rate: {matched.mean()*100:.1f}%')

df_mapped = df[matched].copy()
print(f'\nRows after mapping filter: {len(df_mapped):,}')
print(f'Unique drug pairs with DrugBank IDs: {df_mapped[["drug_1_drugbank_id", "drug_2_drugbank_id"]].drop_duplicates().shape[0]:,}')

## Step 4: Group MedDRA Conditions into Reaction Categories
TwoSIDES has 20,000+ individual MedDRA adverse event terms. We group these into 12 clinically meaningful reaction categories using keyword matching.

These categories become the edge label dimensions in the GNN (one severity score per drug pair per category).

In [ ]:
print('Grouping conditions into reaction categories...')

reaction_categories = {
    'bleeding': [
        'haemorrhage', 'hemorrhage', 'bleeding', 'haematoma', 'hematoma',
        'epistaxis', 'haemoptysis', 'haematuria', 'purpura', 'ecchymosis',
        'prothrombin', 'coagulation', 'thrombocytopenia', 'anticoagulant',
        'deep vein thrombosis', 'thrombosis', 'thromboembolism', 'anaemia',
        'haemoglobin', 'neutropenia', 'pancytopenia', 'platelet count decreased',
        'haematocrit decreased', 'international normalised ratio',
        'white blood cell count decreased', 'leukopenia'
    ],
    'hepatotoxicity': [
        'hepatic', 'liver', 'hepatitis', 'jaundice', 'bilirubin',
        'transaminase', 'alanine aminotransferase', 'aspartate aminotransferase',
        'cholestasis', 'hepatocellular', 'cirrhosis', 'alt increased', 'ast increased',
        'cholelithiasis', 'blood alkaline phosphatase increased'
    ],
    'cardiac': [
        'cardiac', 'heart', 'arrhythmia', 'tachycardia', 'bradycardia',
        'atrial fibrillation', 'myocardial', 'qt prolongation', 'palpitation',
        'ventricular', 'angina', 'coronary', 'cardiotoxic', 'hypertension',
        'hypotension', 'chest pain', 'oedema peripheral', 'pleural effusion',
        'chest discomfort', 'blood pressure increased', 'blood pressure decreased',
        'cardiomegaly', 'cerebrovascular accident', 'hyperlipidaemia'
    ],
    'cns': [
        'seizure', 'convulsion', 'encephalopathy', 'confusion', 'hallucination',
        'serotonin syndrome', 'neuroleptic', 'tremor', 'neuropathy',
        'somnolence', 'sedation', 'coma', 'neurotoxic', 'dizziness', 'syncope',
        'anxiety', 'depression', 'insomnia', 'headache', 'hypoaesthesia',
        'loss of consciousness', 'gait disturbance', 'muscular weakness',
        'paraesthesia', 'mental status changes', 'memory impairment',
        'balance disorder', 'dysarthria', 'lethargy', 'depressed level of consciousness',
        'emotional distress', 'anhedonia', 'stress'
    ],
    'renal': [
        'renal', 'kidney', 'nephrotoxic', 'creatinine', 'nephropathy',
        'acute kidney', 'renal failure', 'glomerular', 'proteinuria',
        'hypokalaemia', 'hyponatraemia', 'dehydration', 'urinary tract',
        'hyperkalaemia', 'hypoglycaemia', 'blood glucose increased',
        'hyperglycaemia', 'diabetes mellitus', 'type 2 diabetes'
    ],
    'gastrointestinal': [
        'diarrhoea', 'vomiting', 'nausea', 'constipation', 'abdominal',
        'dysphagia', 'gastrointestinal', 'colitis', 'gastric', 'bowel',
        'intestinal', 'pancreatitis', 'peptic', 'dyspepsia', 'gastritis',
        'gastrooesophageal reflux'
    ],
    'respiratory': [
        'dyspnoea', 'respiratory', 'pneumonia', 'cough', 'bronchitis',
        'pulmonary', 'asthma', 'respiratory failure', 'hypoxia', 'apnoea',
        'sinusitis'
    ],
    'systemic': [
        'pyrexia', 'fatigue', 'asthenia', 'pain', 'sepsis', 'infection',
        'weight decreased', 'decreased appetite', 'oedema', 'fall',
        'injury', 'general physical health deterioration', 'condition aggravated',
        'cellulitis', 'arthralgia', 'back pain', 'pain in extremity',
        'malaise', 'chills', 'myalgia', 'muscle spasms', 'weight increased',
        'septic shock', 'contusion', 'white blood cell count increased'
    ],
    'dermatological': [
        'rash', 'erythema', 'pruritus', 'hyperhidrosis', 'skin',
        'dermatitis', 'urticaria', 'alopecia', 'sweating'
    ],
    'musculoskeletal': [
        'osteoarthritis', 'arthritis', 'myalgia', 'muscle spasm',
        'bone', 'fracture', 'joint', 'musculoskeletal'
    ],
    'endocrine': [
        'hypothyroidism', 'hyperthyroidism', 'thyroid', 'adrenal',
        'diabetes', 'insulin', 'glucose', 'hypoglycaemia', 'hyperglycaemia'
    ],
    'ophthalmological': [
        'vision blurred', 'eye', 'ocular', 'visual', 'retinal', 'cataract'
    ]
}

def assign_category(condition_name):
    if pd.isna(condition_name):
        return 'other'
    condition_lower = condition_name.lower()
    for category, keywords in reaction_categories.items():
        if any(kw in condition_lower for kw in keywords):
            return category
    return 'other'

df_mapped['reaction_category'] = df_mapped['condition_concept_name'].apply(assign_category)

print('\nCondition distribution across categories:')
print(df_mapped['reaction_category'].value_counts())
print(f"\nRows labelled as 'other': {(df_mapped['reaction_category'] == 'other').sum():,}")

## Step 5: Drop 'Other' and Aggregate to Edge Labels
Remove uncategorisable conditions (noise like 'Economic problem', 'Dental caries') and aggregate to one max PRR score per drug pair × reaction category.

This is the final edge label table — one row per (drug_1, drug_2, reaction_category) triple.

In [ ]:
print("Dropping uncategorised conditions...")
df_clean = df_mapped[df_mapped['reaction_category'] != 'other'].copy()
print(f'Rows remaining: {len(df_clean):,}')

print('\nAggregating to edge labels...')
edge_labels = df_clean.groupby(
    ['drug_1_drugbank_id', 'drug_2_drugbank_id', 'reaction_category']
)['PRR'].max().reset_index()

edge_labels.columns = ['drug_1_drugbank_id', 'drug_2_drugbank_id', 'reaction_category', 'PRR_max']

print(f'Final edge label rows: {len(edge_labels):,}')
print(f'Unique drug pairs with labels: {edge_labels[["drug_1_drugbank_id", "drug_2_drugbank_id"]].drop_duplicates().shape[0]:,}')
print(f'\nReaction category breakdown:')
print(edge_labels['reaction_category'].value_counts())
print(f'\nSample output:')
print(edge_labels.head(10))

## Step 6: Save Output
Save the final edge label table to CSV. This file is the processed TwoSIDES output ready to be joined onto the DrugBank interaction graph.

In [ ]:
edge_labels.to_csv('twosides_edge_labels.csv', index=False)
print('Saved to twosides_edge_labels.csv')
print(f'\nFinal summary:')
print(f'  Rows: {len(edge_labels):,}')
print(f'  Unique drug pairs: {edge_labels[["drug_1_drugbank_id", "drug_2_drugbank_id"]].drop_duplicates().shape[0]:,}')
print(f'  Reaction categories: {edge_labels["reaction_category"].nunique()}')
print(f'  PRR range: {edge_labels["PRR_max"].min():.2f} – {edge_labels["PRR_max"].max():.2f}')